# 🧪 A/B Testing — Uji Statistik
## Twitter Sentiment Analysis — Mental Health Topic

**Isi notebook ini:**
- Uji Normalitas (Shapiro-Wilk)
- Mann-Whitney U Test (2 kelompok)
- Kruskal-Wallis H Test (3 kelompok)
- Section 7: Explanatory Analysis — Jawaban Business Questions

> **Prasyarat:** Jalankan `01_data_wrangling.ipynb` terlebih dahulu.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import ast
import json
import re
import warnings
from collections import Counter
from pathlib import Path

import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu, shapiro

matplotlib.use('Agg')
warnings.filterwarnings('ignore')

In [2]:
# ── Konfigurasi ──────────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../datasets/processed')
OUTPUT_DIR    = Path('../reports/figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05  # Tingkat signifikansi

PAL = {'Positive': '#00C896', 'Neutral': '#4A9EFF', 'Negative': '#FF5A5A'}
THEME = {
    'BG'  : '#0D1117',
    'CARD': '#161B22',
    'GRID': '#21262D',
    'TXT' : '#E6EDF3',
    'SUB' : '#8B949E',
    'YEL' : '#F0C040',
}

plt.rcParams.update({
    'figure.facecolor' : THEME['BG'],
    'axes.facecolor'   : THEME['CARD'],
    'axes.edgecolor'   : THEME['GRID'],
    'text.color'       : THEME['TXT'],
    'xtick.color'      : THEME['SUB'],
    'ytick.color'      : THEME['SUB'],
    'grid.color'       : THEME['GRID'],
    'axes.labelcolor'  : THEME['TXT'],
    'axes.titlecolor'  : THEME['TXT'],
    'legend.facecolor' : THEME['CARD'],
    'legend.edgecolor' : THEME['GRID'],
    'legend.labelcolor': THEME['TXT'],
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.titlepad'    : 10,
})

In [3]:
# ── Helper Functions ─────────────────────────────────────────────────────────
def format_pvalue(p: float) -> str:
    """Format p-value: tampilkan 6 desimal atau '< 0.0001' jika sangat kecil."""
    return f'{p:.6f}' if p >= 0.0001 else '< 0.0001'


def is_significant(p: float, alpha: float = ALPHA) -> str:
    """Kembalikan label keputusan uji hipotesis."""
    return f'✅ SIGNIFIKAN → Tolak H₀ (p={format_pvalue(p)})' if p < alpha \
        else f'❌ Tidak Signifikan → Gagal Tolak H₀ (p={format_pvalue(p)})'


def save_figure(fig: plt.Figure, name: str) -> None:
    """Simpan figure ke OUTPUT_DIR dengan format PNG."""
    path = OUTPUT_DIR / f'{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=THEME['BG'])
    plt.close(fig)
    print(f'✅ Disimpan → {path}')


def parse_hashtags(raw: str) -> list[str]:
    """Re-parse hashtag dari string representasi Python list."""
    try:
        items = ast.literal_eval(str(raw))
        return [
            re.sub(r"['\"\s#]", '', str(item)).lower().strip()
            for item in items
            if len(str(item).strip()) > 1
        ]
    except Exception:
        return []

In [4]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv(PROCESSED_DIR / 'twitter_clean.csv')

with open(PROCESSED_DIR / 'top_hashtags.json') as f:
    top_hashtags = json.load(f)

if 'hashtag_list' not in df.columns:
    df['hashtag_list'] = df['hashtags'].apply(parse_hashtags)

pos = df[df['sentiment'] == 'Positive']
neu = df[df['sentiment'] == 'Neutral']
neg = df[df['sentiment'] == 'Negative']

print(f'✅ Dataset dimuat: {len(df):,} baris')

✅ Dataset dimuat: 8,439 baris


---
## A/B Testing — Uji Statistik

Sebelum memilih uji parametrik/non-parametrik, periksa asumsi normalitas terlebih dahulu.

In [5]:
# Uji Normalitas (Shapiro-Wilk) — sampel 200 per kelompok untuk efisiensi
SAMPLE_SIZE = 200

print(f'Uji Normalitas Shapiro-Wilk (n={SAMPLE_SIZE} per kelompok)\n')
for sentiment, group in [('Positive', pos), ('Neutral', neu), ('Negative', neg)]:
    sample = group['hashtag_count'].sample(min(SAMPLE_SIZE, len(group)), random_state=42)
    stat, p = shapiro(sample)
    print(f'  {sentiment:<10} stat={stat:.4f}  {is_significant(p)}')

Uji Normalitas Shapiro-Wilk (n=200 per kelompok)

  Positive   stat=0.6991  ✅ SIGNIFIKAN → Tolak H₀ (p=< 0.0001)
  Neutral    stat=0.7737  ✅ SIGNIFIKAN → Tolak H₀ (p=< 0.0001)
  Negative   stat=0.7933  ✅ SIGNIFIKAN → Tolak H₀ (p=< 0.0001)


In [6]:
# BQ3 — Mann-Whitney U Test: Hashtag Count (Positive vs Negative)
# H0: Tidak ada perbedaan hashtag count antara Positive dan Negative
u_ht, p_ht = mannwhitneyu(pos['hashtag_count'], neg['hashtag_count'], alternative='two-sided')

print('BQ3 — Hashtag Count: Positive vs Negative')
print(f'  Mean Positive : {pos["hashtag_count"].mean():.3f}')
print(f'  Mean Negative : {neg["hashtag_count"].mean():.3f}')
print(f'  U-statistic   : {u_ht:.2f}')
print(f'  Keputusan     : {is_significant(p_ht)}')

BQ3 — Hashtag Count: Positive vs Negative
  Mean Positive : 3.728
  Mean Negative : 5.247
  U-statistic   : 2713476.00
  Keputusan     : ✅ SIGNIFIKAN → Tolak H₀ (p=< 0.0001)


In [7]:
# BQ3 — Mann-Whitney U Test: Text Length (Positive vs Negative)
# H0: Tidak ada perbedaan text length antara Positive dan Negative
u_tl2, p_tl2 = mannwhitneyu(pos['text_length'], neg['text_length'], alternative='two-sided')

print('BQ3 — Text Length: Positive vs Negative')
print(f'  Mean Positive : {pos["text_length"].mean():.2f}')
print(f'  Mean Negative : {neg["text_length"].mean():.2f}')
print(f'  U-statistic   : {u_tl2:.2f}')
print(f'  p-value       : {p_tl2:.6f}')
print(f'  Keputusan     : {is_significant(p_tl2)}')


BQ3 — Text Length: Positive vs Negative
  Mean Positive : 180.03
  Mean Negative : 182.05
  U-statistic   : 3088198.00
  p-value       : 0.716415
  Keputusan     : ❌ Tidak Signifikan → Gagal Tolak H₀ (p=0.716415)


In [8]:
# BQ4 — Kruskal-Wallis: Hashtag Count & Text Length (semua sentimen)
# H0: Tidak ada perbedaan antar ketiga kelompok sentimen
h_ht_all, p_ht_all = kruskal(pos['hashtag_count'], neu['hashtag_count'], neg['hashtag_count'])
h_tl, p_tl = kruskal(pos['text_length'], neu['text_length'], neg['text_length'])

print('BQ4 — Hashtag Count (Positive vs Neutral vs Negative)')
print(f'  H-statistic   : {h_ht_all:.4f}')
print(f'  p-value       : {p_ht_all:.6f}')
print(f'  Keputusan     : {is_significant(p_ht_all)}')
print()
print('BQ4 — Text Length (Positive vs Neutral vs Negative)')
print(f'  H-statistic   : {h_tl:.4f}')
print(f'  p-value       : {p_tl:.6f}')
print(f'  Keputusan     : {is_significant(p_tl)}')


BQ4 — Hashtag Count (Positive vs Neutral vs Negative)
  H-statistic   : 101.0605
  p-value       : 0.000000
  Keputusan     : ✅ SIGNIFIKAN → Tolak H₀ (p=< 0.0001)

BQ4 — Text Length (Positive vs Neutral vs Negative)
  H-statistic   : 1.0408
  p-value       : 0.594291
  Keputusan     : ❌ Tidak Signifikan → Gagal Tolak H₀ (p=0.594291)


In [9]:
# Simpan hasil uji statistik ke JSON
ab_results = {
    'hashtag_count_pos_neg': {
        'test'       : 'Mann-Whitney U',
        'metric'     : 'Hashtag Count',
        'group_a'    : 'Positive',
        'group_b'    : 'Negative',
        'mean_a'     : round(pos['hashtag_count'].mean(), 3),
        'mean_b'     : round(neg['hashtag_count'].mean(), 3),
        'statistic'  : round(u_ht, 2),
        'pvalue'     : float(p_ht),
        'significant': bool(p_ht < ALPHA),
        'h0'         : 'Tidak ada perbedaan hashtag count antara Positive dan Negative',
    },
    'text_length_pos_neg': {
        'test'       : 'Mann-Whitney U',
        'metric'     : 'Text Length (Pos vs Neg)',
        'group_a'    : 'Positive',
        'group_b'    : 'Negative',
        'mean_a'     : round(pos['text_length'].mean(), 2),
        'mean_b'     : round(neg['text_length'].mean(), 2),
        'statistic'  : round(u_tl2, 2),
        'pvalue'     : float(p_tl2),
        'significant': bool(p_tl2 < ALPHA),
        'h0'         : 'Tidak ada perbedaan text length antara Positive dan Negative',
    },
    'hashtag_count_all': {
        'test'       : 'Kruskal-Wallis',
        'metric'     : 'Hashtag Count (semua sentimen)',
        'groups'     : ['Positive', 'Neutral', 'Negative'],
        'means'      : [round(pos['hashtag_count'].mean(), 3),
                        round(neu['hashtag_count'].mean(), 3),
                        round(neg['hashtag_count'].mean(), 3)],
        'statistic'  : round(h_ht_all, 4),
        'pvalue'     : float(p_ht_all),
        'significant': bool(p_ht_all < ALPHA),
        'h0'         : 'Tidak ada perbedaan hashtag count antar ketiga sentimen',
    },
    'text_length_all': {
        'test'       : 'Kruskal-Wallis',
        'metric'     : 'Text Length (semua sentimen)',
        'groups'     : ['Positive', 'Neutral', 'Negative'],
        'means'      : [round(pos['text_length'].mean(), 2),
                        round(neu['text_length'].mean(), 2),
                        round(neg['text_length'].mean(), 2)],
        'statistic'  : round(h_tl, 4),
        'pvalue'     : float(p_tl),
        'significant': bool(p_tl < ALPHA),
        'h0'         : 'Tidak ada perbedaan text length antar ketiga sentimen',
    },
}

out_path = PROCESSED_DIR / 'ab_results.json'
with open(out_path, 'w') as f:
    json.dump(ab_results, f, indent=2)

print(f'✅ Hasil A/B Testing disimpan → {out_path}')


✅ Hasil A/B Testing disimpan → ..\datasets\processed\ab_results.json


In [10]:
# Viz 4 — A/B Testing Violin Plot
BG   = THEME['BG']
CARD = THEME['CARD']
GRID = THEME['GRID']
TXT  = THEME['TXT']
SUB  = THEME['SUB']

tests_config = [
    ('hashtag_count', 'BQ3 — Hashtag Count',       'Jumlah Hashtag', p_ht,  ['Positive', 'Negative']),
    ('text_length',   'BQ3 — Text Length Pos/Neg',  'Panjang Teks',   p_tl2, ['Positive', 'Negative']),
    ('hashtag_count', 'BQ4 — Hashtag Count (All)',  'Jumlah Hashtag', p_ht_all, ['Positive', 'Neutral', 'Negative']),
    ('text_length',   'BQ4 — Text Length (All)',    'Panjang Teks',   p_tl,  ['Positive', 'Neutral', 'Negative']),
]

fig4, axes4 = plt.subplots(1, 4, figsize=(22, 6), facecolor=BG)
fig4.suptitle(
    f'A/B Testing — Distribusi Fitur per Sentimen  |  alpha = {ALPHA}',
    fontsize=17, fontweight='bold', color=TXT, y=1.02,
)

for ax, (col, title, ylabel, pval, groups) in zip(axes4, tests_config):
    ax.set_facecolor(CARD)
    data = [df[df['sentiment'] == s][col].values for s in groups]
    positions = list(range(1, len(groups) + 1))

    parts = ax.violinplot(data, positions=positions, showmedians=True, showextrema=False)
    for body, sentiment in zip(parts['bodies'], groups):
        body.set_facecolor(PAL[sentiment])
        body.set_alpha(0.75)
        body.set_edgecolor(BG)
    parts['cmedians'].set_color('white')
    parts['cmedians'].set_linewidth(2.5)

    ax.set_xticks(positions)
    ax.set_xticklabels([g[:3] for g in groups], color=TXT, fontsize=10)
    ax.set_ylabel(ylabel, color=SUB, fontsize=9)

    sig_label = 'SIGNIFIKAN' if pval < ALPHA else 'Tidak Signifikan'
    ax.set_title(f'{title}\np = {format_pvalue(pval)}  [{sig_label}]', fontsize=10.5, color=TXT, pad=8)
    ax.grid(axis='y', alpha=0.2, color=GRID)

plt.tight_layout()
save_figure(fig4, 'viz4_ab_testing')


✅ Disimpan → ..\reports\figures\viz4_ab_testing.png


---
## 7. Explanatory Analysis — Jawaban Business Questions

In [11]:
# Viz 5 — Explanatory Analysis Dashboard (2×3 grid)
YEL = THEME['YEL']

fig5 = plt.figure(figsize=(22, 13), facecolor=BG)
fig5.suptitle(
    'Explanatory Analysis — Jawaban 5 Business Questions',
    fontsize=22, fontweight='bold', color=TXT, y=0.98,
)
gs5 = gridspec.GridSpec(2, 3, figure=fig5, hspace=0.5, wspace=0.38)

# Panel 5A: Distribusi Sentimen (BQ1)
ax_5a = fig5.add_subplot(gs5[0, 0])
sizes  = [len(pos), len(neu), len(neg)]
labels = ['Positive\n(17.8%)', 'Neutral\n(33.2%)', 'Negative\n(48.9%)']
ax_5a.pie(
    sizes, labels=labels, colors=list(PAL.values()),
    autopct='%1.1f%%', startangle=140,
    textprops={'color': TXT, 'fontsize': 9},
    wedgeprops={'edgecolor': BG, 'linewidth': 2},
    pctdistance=0.75,
)
ax_5a.set_title(
    'BQ1 — Distribusi Sentimen Pasca-Cleaning\n'
    '(Jawaban: Distribusi TIDAK seimbang!)',
    fontsize=11, fontweight='bold', pad=8, color=YEL,
)

# Panel 5B: Text Length distribution (BQ4)
ax_5b = fig5.add_subplot(gs5[0, 1])
for sentiment in ['Positive', 'Neutral', 'Negative']:
    vals = df[df['sentiment'] == sentiment]['text_length'].clip(upper=300)
    ax_5b.hist(vals, bins=40, alpha=0.55, color=PAL[sentiment], label=sentiment,
               density=True, range=(0, 300), edgecolor=BG, linewidth=0.5)
ax_5b.set_xlabel('Panjang Teks (maks. 300 karakter)')
ax_5b.set_ylabel('Densitas')
ax_5b.set_title(
    'BQ4 — Distribusi Text Length\n(Jawaban: Tidak berbeda signifikan!)',
    fontsize=11, fontweight='bold', pad=8, color=TXT,
)
ax_5b.legend(fontsize=9)
ax_5b.grid(alpha=0.2)

# Panel 5C: Ringkasan A/B Testing
ax_5c = fig5.add_subplot(gs5[0, 2])
ax_5c.axis('off')
table_data = [
    ['Variabel', 'Test', 'p-value', 'Signifikan?'],
    ['Hashtag Count\n(Pos vs Neg)', 'Mann-Whitney', '< 0.0001', '✅ Ya'],
    ['Text Length\n(Pos vs Neg)', 'Mann-Whitney', '> 0.05', '❌ Tidak'],
    ['Hashtag Count\n(All)', 'Kruskal-Wallis', '< 0.0001', '✅ Ya'],
    ['Text Length\n(All)', 'Kruskal-Wallis', '> 0.05', '❌ Tidak'],
]
table = ax_5c.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    cellLoc='center',
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.3, 2.0)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor(GRID)
    cell.set_facecolor(CARD if row > 0 else '#21262D')
    cell.set_text_props(color=TXT)
ax_5c.set_title('BQ3 & BQ4 — Ringkasan Uji Statistik', fontsize=11, fontweight='bold', pad=10)

# Panel 5D: Distribusi Hashtag Count (BQ3)
ax_5d = fig5.add_subplot(gs5[1, 0])
for sentiment in ['Positive', 'Neutral', 'Negative']:
    vals = df[df['sentiment'] == sentiment]['hashtag_count'].clip(upper=20)
    ax_5d.hist(vals, bins=20, alpha=0.55, color=PAL[sentiment], label=sentiment,
               density=True, range=(0, 20), edgecolor=BG, linewidth=0.5)
for sentiment in ['Positive', 'Negative']:
    mean_val = df[df['sentiment'] == sentiment]['hashtag_count'].mean()
    ax_5d.axvline(mean_val, color=PAL[sentiment], linestyle='--', linewidth=1.5, alpha=0.9)
    ax_5d.text(mean_val + 0.15, ax_5d.get_ylim()[1] * 0.85,
               f'μ={mean_val:.1f}', color=PAL[sentiment], fontsize=9, fontweight='bold')
ax_5d.set_xlabel('Jumlah Hashtag (maks. 20)')
ax_5d.set_ylabel('Densitas')
ax_5d.set_title(
    'BQ3 — Distribusi Hashtag Count\n(Jawaban: Tweet Negatif lebih banyak hashtag!)',
    fontsize=11, fontweight='bold', pad=8, color=YEL,
)
ax_5d.legend(fontsize=9)
ax_5d.grid(alpha=0.2)

# Panel 5E: Ironi Digital (BQ5)
ax_5e = fig5.add_subplot(gs5[1, 1])
ironic_tags = ['happy', 'excited', 'love', 'fun', 'great', 'amazing', 'blessed']
neg_tags_flat = [
    tag
    for tag_list in df[df['sentiment'] == 'Negative']['hashtag_list']
    for tag in tag_list
]
neg_counter = Counter(neg_tags_flat)
ironic_counts = {tag: neg_counter[tag] for tag in ironic_tags if tag in neg_counter}

if ironic_counts:
    tags_ir   = [f'#{tag}' for tag in ironic_counts]
    counts_ir = list(ironic_counts.values())
    bars_ir = ax_5e.barh(tags_ir, counts_ir, color=PAL['Positive'], alpha=0.8,
                          edgecolor=BG, height=0.6)
    for bar in bars_ir:
        ax_5e.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
                   f'{int(bar.get_width()):,}', va='center', fontsize=9.5, color=TXT)
    ax_5e.set_xlabel('Frekuensi dalam Tweet NEGATIF')
    ax_5e.set_title(
        'BQ5 — Ironi Digital\nHashtag Positif dalam Tweet Negatif',
        fontsize=11, fontweight='bold', pad=8, color=PAL['Negative'],
    )
    ax_5e.text(
        0.98, 0.05,
        'Fenomena IRONI:\nPengguna mengekspresikan\nfrustasi menggunakan\nkosakata positif',
        transform=ax_5e.transAxes, ha='right', va='bottom',
        fontsize=8, color=YEL,
        bbox=dict(boxstyle='round,pad=0.4', facecolor=CARD, edgecolor=YEL, alpha=0.8),
    )
    ax_5e.grid(axis='x', alpha=0.2)

# Panel 5F: Ringkasan insight
ax_5f = fig5.add_subplot(gs5[1, 2])
ax_5f.set_facecolor(CARD)
ax_5f.axis('off')

insights = [
    ('BQ1', PAL['Neutral'],   'Distribusi Sentimen',
     'Raw seimbang (33.3%).\nPasca dedup: Neg 48.9%, Neu 33.2%, Pos 17.8%'),
    ('BQ2', PAL['Positive'],  'Hashtag Dominan',
     '#mentalhealth & #stress di Pos & Neu.\n#happy & #excited di Neg (ironi!)'),
    ('BQ3', PAL['Negative'],  'Hashtag Count [SIGNIFIKAN]',
     'p < 0.0001: Neg (5.25) vs Pos (3.73)\n→ 40.8% lebih banyak hashtag di tweet Neg'),
    ('BQ4', THEME['SUB'],     'Panjang Teks [tidak signifikan]',
     'p > 0.05: Text length tidak membedakan\nsentimen secara signifikan'),
    ('BQ5', THEME['YEL'],     'Ironi Digital',
     '#happy muncul 1.012x, #excited 757x\ndalam tweet bersentimen NEGATIF'),
]

y_pos = 0.97
for code, color, title, desc in insights:
    ax_5f.text(0.02, y_pos, f'● {code}: {title}',
               transform=ax_5f.transAxes, fontsize=9.5, fontweight='bold',
               color=color, va='top')
    y_pos -= 0.055
    for line in desc.split('\n'):
        ax_5f.text(0.06, y_pos, line, transform=ax_5f.transAxes,
                   fontsize=8.5, color=TXT, va='top')
        y_pos -= 0.042
    y_pos -= 0.025

ax_5f.set_title('Ringkasan Insight & Jawaban BQ', fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
save_figure(fig5, 'viz5_explanatory')


✅ Disimpan → ..\reports\figures\viz5_explanatory.png
